### First few cells is just data preparation

My implementation is build on top of nequip and allegro python libraries to simplify data load and preprocessing.
The wigner simbols are from e3nn
Here are the links
https://github.com/mir-group/nequip
https://github.com/mir-group/allegro

In [1]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config
import os

default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cuda:0',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)
import numpy as np
import random
import torch
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    #print(f"Random seed set as {seed}")

os.environ['NEQUIP_NUM_TASKS'] = '16'
# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [2]:
config = Config.from_file('./config/example_ETN_opt_MEA.yaml', defaults=default_config)

config['root'] = 'results/MEA_Allegro_2'
config['seed'] = 123456 + 8
set_seed(config['seed'])
torch.manual_seed(config['seed'])
dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

In [3]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# Some hyperparameteres
#Nc = 10 # number of chennels for F features from ETN paper
#N_rank_spec = 4 # hidden rank of reduction for type radial tensor
#config['Nc'] = Nc
#config['N_rank_spec'] = N_rank_spec

# ETN parameters
#config['d'] = 4 # dimention of the tensor train
#config['N_rank_ett'] = [4, 4, 4] # ranks of tensor train



# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

DEBUG:root:* Initialize Output
  ...generate file name results/MEA_Allegro_2/example/log
  ...open log file results/MEA_Allegro_2/example/log
  ...generate file name results/MEA_Allegro_2/example/metrics_epoch.csv
  ...open log file results/MEA_Allegro_2/example/metrics_epoch.csv
  ...generate file name results/MEA_Allegro_2/example/metrics_initialization.csv
  ...open log file results/MEA_Allegro_2/example/metrics_initialization.csv
  ...generate file name results/MEA_Allegro_2/example/metrics_batch_train.csv
  ...open log file results/MEA_Allegro_2/example/metrics_batch_train.csv
  ...generate file name results/MEA_Allegro_2/example/metrics_batch_val.csv
  ...open log file results/MEA_Allegro_2/example/metrics_batch_val.csv
  ...generate file name results/MEA_Allegro_2/example/best_model.pth
  ...generate file name results/MEA_Allegro_2/example/last_model.pth
  ...generate file name results/MEA_Allegro_2/example/trainer.pth
  ...generate file name results/MEA_Allegro_2/example/config

In [4]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

trainer.model = final_model

# Test configuration stores as dict of parameters
data0 = AtomicData.to_AtomicDataDict(dataset[0])

In [5]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math
# forward pass
data_new = final_model(data0)



In [6]:
data_new['atomic_energy'].max()

tensor(-13.2993, grad_fn=<MaxBackward1>)

In [7]:
trainer.train()

Number of weights: 4920
Number of trainable weights: 4920
instantiate Adam
        all_args :                                               betas <-                             optimizer_params.betas
        all_args :                                             amsgrad <-                           optimizer_params.amsgrad
        all_args :                                                 eps <-                               optimizer_params.eps
        all_args :                                        weight_decay <-                      optimizer_params.weight_decay
...Adam_param = dict(
...   optional_args = {'betas': (0.9, 0.999), 'eps': 1e-08, 'weight_decay': 0.0, 'amsgrad': False, 'foreach': None, 'maximize': False, 'capturable': False, 'differentiable': False, 'fused': None},
...   positional_args = {'params': <generator object Module.parameters at 0x7fd220d73610>, 'lr': 0.001})
instantiate ReduceLROnPlateau
        all_args :                                            patience 

In [ ]:
1   100         1.42         1.29        0.127        0.547        0.975         5.25        0.243
1   100         3.23         2.51        0.716        0.673         1.36         10.3        0.556


In [15]:
config

{'_jit_bailout_depth': 2, '_jit_fusion_strategy': [('DYNAMIC', 3)], '_jit_fuser': 'fuser1', 'root': 'results/MEA_Allegro_1', 'tensorboard': False, 'wandb': False, 'model_builders': ['allegro.model.ETN_opt', 'PerSpeciesRescale', 'ParaStressForceOutput', 'RescaleEnergyEtc'], 'dataset_statistics_stride': 1, 'device': 'cuda:0', 'default_dtype': 'float32', 'model_dtype': 'float32', 'allow_tf32': True, 'verbose': 'debug', 'model_debug_mode': False, 'equivariance_test': False, 'grad_anomaly_mode': False, 'gpu_oom_offload': False, 'append': True, 'warn_unused': False, 'run_name': 'example', 'seed': 123458, 'dataset_seed': 123456, 'r_max': 5.0, 'Nc': 10, 'd': 4, 'N_rank_spec': 4, 'N_rank_ett': [4, 4, 4], 'avg_num_neighbors': 26.18090057373047, 'BesselBasis_trainable': True, 'PolynomialCutoff_p': 5, 'l_max': 2, 'parity': 'o3_full', 'num_layers': 2, 'env_embed_multiplicity': 64, 'embed_initial_edge': True, 'two_body_latent_mlp_latent_dimensions': [128, 256, 512, 1024], 'two_body_latent_mlp_nonlin

In [10]:
def batch_step(data, validation=False):
    # no need to have gradients from old steps taking up memory
    #self.optim.zero_grad(set_to_none=True)

    #if validation:
    #    self.model.eval()
    #else:
    #    self.model.train()

    # Do any target rescaling
    data = AtomicData.to_AtomicDataDict(data)

    # this will normalize the targets
    # in both validation and train we want targets normalized _for the loss_
    data_for_loss = trainer.model.unscale(data, force_process=True)

    # Run model
    # We make a shallow copy of the input dict in case the model modifies it
    out = trainer.model(data_for_loss)
    #print(out)
    return out

In [12]:
from nequip.train._key import ABBREV, LOSS_KEY, TRAIN, VALIDATION

def epoch_step(trainer):

    dataloaders = {TRAIN: trainer.dl_train, VALIDATION: trainer.dl_val}
    categories = [TRAIN, VALIDATION] if trainer.iepoch >= 0 else [VALIDATION]
    dataloaders = [
        dataloaders[c] for c in categories
    ]  # get the right dataloaders for the catagories we actually run
    if TRAIN in categories:
        # We have to step the sampler so it knows what epoch it is
        trainer.dl_train_sampler.step_epoch(trainer.iepoch)

    #self.metrics_dict = {}
    #self.loss_dict = {}

    for category, dataset in zip(categories, dataloaders):
        
        for trainer.ibatch, batch in enumerate(dataset):
            print(trainer.ibatch, batch)
            out = batch_step(
                data=batch,
                validation=(category == VALIDATION),
            )
            
    return out, batch

In [13]:
len(dataset[:20])

20

In [14]:
dataset[:20]

ASEDataset(20)

In [15]:
trainer.n_train

5000

In [16]:
trainer.dl_val.batch_size

5

In [19]:
trainer.n_train = 2
trainer.n_val = 3

trainer.train_idcs = torch.tensor([201, 1201], dtype = torch.long)
trainer.val_idcs = torch.tensor([301, 1401, 5], dtype = torch.long)

trainer.set_dataset(dataset, None)

In [20]:
out, batch = epoch_step(trainer)

0 Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])


In [21]:
out['pos'].shape

torch.Size([34, 3])

In [22]:
out['batch']

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 2, 2])

In [23]:
out['ptr']

tensor([ 0, 16, 32, 34])

In [24]:
print(AtomicData.to_AtomicDataDict(dataset[301])['pos'].shape)
print(AtomicData.to_AtomicDataDict(dataset[1401])['pos'].shape)
print(AtomicData.to_AtomicDataDict(dataset[5])['pos'].shape)

torch.Size([16, 3])
torch.Size([16, 3])
torch.Size([2, 3])


In [25]:
out.keys()

dict_keys(['edge_index', 'pos', 'batch', 'ptr', 'cell', 'edge_cell_shift', 'atom_types', 'edge_vectors', 'edge_types', 'node_attrs', 'node_features', 'edge_lengths', 'edge_embedding', 'edge_cutoff', 'edge_attrs', 'edge_features_F', 'node_features_F', 'node_features_ETN', 'atomic_energy', 'total_energy', 'forces', 'stress', 'virial', 'atom_virial'])

In [28]:
trainer.batch_metrics = trainer.metrics(pred=out, ref=batch)

ValueError: Data shape of batch, torch.Size([3, 34]), does not match the input data dimension of this RunningStats, torch.Size([192])

In [29]:
batch

Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])

In [30]:
for key in out:
    print(key, out[key].shape)

edge_index torch.Size([2, 884])
pos torch.Size([34, 3])
batch torch.Size([34])
ptr torch.Size([4])
cell torch.Size([3, 3, 3])
edge_cell_shift torch.Size([884, 3])
atom_types torch.Size([34, 1])
edge_vectors torch.Size([884, 3])
edge_types torch.Size([884, 1])
node_attrs torch.Size([34, 4])
node_features torch.Size([34, 4])
edge_lengths torch.Size([884])
edge_embedding torch.Size([884, 8])
edge_cutoff torch.Size([884, 1])
edge_attrs torch.Size([884, 9])
edge_features_F torch.Size([884, 9, 10])
node_features_F torch.Size([34, 9, 10])
node_features_ETN torch.Size([34, 9, 10])
atomic_energy torch.Size([34, 34])
total_energy torch.Size([3, 34])
forces torch.Size([34, 3])
stress torch.Size([3, 3, 3])
virial torch.Size([3, 3, 3])
atom_virial torch.Size([34, 3, 3])


In [31]:
batch

Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])

In [32]:
out['total_energy'].shape

torch.Size([3, 34])

In [40]:

( data_new[_keys.NODE_FEATURES_ETN] * data_new[_keys.NODE_FEATURES_ETN] ).sum(dim = (-2, -1)).unsqueeze(-1)

tensor([[5.7274e-15],
        [5.7274e-15]], grad_fn=<UnsqueezeBackward0>)

In [37]:
data_new["node_features_F"].shape

torch.Size([2, 9, 10])